# The recommender on a new season: 2026

*Text shortened on 23 September; code and outputs are unchanged. The plan as committed before running is in commit `9c6e5fe`.*

Harder than notebook 14: the 2026 season (January to June), after an off-season, with two
agents and one map that 2025 never had. No 2026 line-up had been looked at before this. Its
dates come from vlr.gg (`fetch_vlr_dates.py`, checked on 2025 first). **Committed to git before
it ran.**

**Decided in advance:**

| | |
|---|---|
| Methods and settings | Exactly notebook 14's, unchanged |
| History | All of 2024 and 2025, then week by week through 2026 |
| Measures, margin of error, verdict | As in notebook 14 |
| Breakdowns | Two only: questions about agents new in 2026, and maps not played in 2025 |
| Sealed | 2026's win results. This notebook reads line-ups only |

Expected: lower than notebook 14. Whether the lead over the naive rule survives is the
finding.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)

from dataset import build_map_table
from recommend import Classifier, CoOccurrence, Popularity, lineup_table, summarise, walk_forward

SEED, RESAMPLES = 0, 5000
HEADLINE, BAR = "gradient boosting", "popularity (naive rule)"
METHODS = {
    "gradient boosting": lambda: Classifier(45, estimator=HistGradientBoostingClassifier(random_state=0)),
    "classifier + map x agent": lambda: Classifier(21, interactions=True),
    "co-occurrence": lambda: CoOccurrence(21),
    "classifier": lambda: Classifier(45),
    "popularity (naive rule)": lambda: Popularity(7),
}

## The line-ups

In [2]:
history = pd.concat([lineup_table(build_map_table("vct_2024")), lineup_table(build_map_table())],
                    ignore_index=True)
to_score = lineup_table(build_map_table("vct_2026"))      # line-ups only -- no results are loaded

print(f"history : {len(history):,} line-ups, {history.played_at.min():%d %b %Y} to {history.played_at.max():%d %b %Y}")
print(f"scored  : {len(to_score):,} line-ups from {to_score.match_id.nunique()} matches, "
      f"{to_score.played_at.min():%d %b} to {to_score.played_at.max():%d %b %Y}, "
      f"over {to_score.played_at.dt.to_period('W').nunique()} weeks")
print(f"          x 5 hidden agents = {len(to_score) * 5:,} complete-a-line-up questions")

known_agents = {a for lineup in history["agents"] for a in lineup}
new_agents = sorted({a for lineup in to_score["agents"] for a in lineup} - known_agents)
maps_2025 = set(history.loc[history["played_at"].dt.year == 2025, "map"])
maps_not_in_2025 = sorted(set(to_score["map"]) - maps_2025)
print(f"\nagents never played before 2026: {new_agents or 'none'}")
print(f"maps not played in 2025: {maps_not_in_2025 or 'none'}")

history : 4,752 line-ups, 17 Feb 2024 to 05 Oct 2025
scored  : 1,772 line-ups from 342 matches, 16 Jan to 21 Jun 2026, over 21 weeks
          x 5 hidden agents = 8,860 complete-a-line-up questions

agents never played before 2026: ['miks', 'veto']
maps not played in 2025: ['Breeze']


## Run every method once

Gradient boosting refits every week, so this takes several minutes.

In [3]:
runs = {name: walk_forward(history, to_score, make) for name, make in METHODS.items()}

## The result

In [4]:
results = pd.DataFrame({name: summarise(r) for name, r in runs.items()}).T
print(f"{len(to_score):,} line-ups from 2026.  Picking at random from the ~23 agents left gets top-3 about 13%.\n")
print(results.to_string(formatters={"top-1": "{:.1%}".format, "top-3": "{:.1%}".format,
                                    "from scratch, of 5": "{:.2f}".format}))

1,772 line-ups from 2026.  Picking at random from the ~23 agents left gets top-3 about 13%.

                         top-1 top-3 from scratch, of 5
gradient boosting        65.7% 87.4%               3.16
classifier + map x agent 67.8% 88.2%               3.16
co-occurrence            67.7% 86.8%               3.21
classifier               63.9% 86.9%               3.13
popularity (naive rule)  49.7% 72.1%               3.30


## The margin of error

In [5]:
def per_lineup(result, cutoff):
    return (np.vstack(result["ranks"].to_numpy()) <= cutoff).mean(axis=1)

match_of = runs[HEADLINE]["match_id"].to_numpy()
match_ids = np.unique(match_of)
rows_of = {m: np.flatnonzero(match_of == m) for m in match_ids}
rng = np.random.default_rng(SEED)
draws = [np.concatenate([rows_of[m] for m in rng.choice(match_ids, len(match_ids))]) for _ in range(RESAMPLES)]

def interval(values):
    return np.percentile([values[d].mean() for d in draws], [2.5, 97.5])

for label, cutoff in [("top-3", 3), ("top-1", 1)]:
    v = per_lineup(runs[HEADLINE], cutoff)
    lo, hi = interval(v)
    print(f"{HEADLINE} {label}: {v.mean():.1%}   95% range {lo:.1%} to {hi:.1%}")

lead = per_lineup(runs[HEADLINE], 3) - per_lineup(runs[BAR], 3)
lead_lo, lead_hi = interval(lead)
print(f"\ntop-3 lead over the naive rule: {lead.mean() * 100:+.1f} points   "
      f"95% range {lead_lo * 100:+.1f} to {lead_hi * 100:+.1f}")

if lead.mean() > 0 and lead_lo > 0:
    verdict = "BEATS THE NAIVE RULE"
elif lead.mean() > 0:
    verdict = "AHEAD, BUT NOT DISTINGUISHABLE FROM THE NAIVE RULE"
else:
    verdict = "DOES NOT BEAT THE NAIVE RULE"
print(f"\nby the rule decided in advance: {verdict}")

gradient boosting top-3: 87.4%   95% range 86.1% to 88.6%
gradient boosting top-1: 65.7%   95% range 64.0% to 67.3%

top-3 lead over the naive rule: +15.3 points   95% range +14.0 to +16.6

by the rule decided in advance: BEATS THE NAIVE RULE


## The two pre-registered breakdowns: what's new

In [6]:
headline_run = runs[HEADLINE]

# 1. questions whose hidden agent had never been played before 2026
questions = pd.DataFrame(
    [(row, agent, rank)
     for row, rank_list in headline_run["ranks"].items()
     for agent, rank in zip(to_score.at[row, "agents"], rank_list)],
    columns=["row", "hidden agent", "rank"],
)
questions["new in 2026"] = questions["hidden agent"].isin(new_agents)
print("1. by whether the hidden agent was new in 2026")
for new, part in questions.groupby("new in 2026"):
    label = "new agent" if new else "agent seen before 2026"
    print(f"   {label:24} {len(part):5} questions   top-1 {(part['rank'] <= 1).mean():.1%}   "
          f"top-3 {(part['rank'] <= 3).mean():.1%}")
if new_agents:
    print("\n   the new agents, week by week (share of questions with the new agent in the top 3):")
    newq = questions[questions["new in 2026"]].assign(week=lambda d: headline_run.loc[d["row"], "week"].to_numpy())
    weekly = newq.groupby("week")["rank"].apply(lambda r: (r <= 3).mean())
    print("   " + "  ".join(f"{w:%d %b} {v:.0%}" for w, v in weekly.items()))

# 2. line-ups on maps not played in 2025
print("\n2. by whether the map was played in 2025")
on_absent = headline_run["map"].isin(maps_not_in_2025)
groups = [(f"{', '.join(maps_not_in_2025)} (not in 2025)", on_absent), ("maps played in 2025", ~on_absent)]
for label, mask in (groups if maps_not_in_2025 else [("every map was played in 2025", ~on_absent)]):
    part = headline_run[mask]
    print(f"   {label:28} {len(part):4} line-ups   " +
          "   ".join(f"{k} {v:.1%}" if k != "from scratch, of 5" else f"{k} {v:.2f}"
                     for k, v in summarise(part).items()))

1. by whether the hidden agent was new in 2026
   agent seen before 2026    8783 questions   top-1 66.0%   top-3 87.8%
   new agent                   77 questions   top-1 29.9%   top-3 42.9%

   the new agents, week by week (share of questions with the new agent in the top 3):
   12 Jan 0%  19 Jan 36%  26 Jan 62%  02 Feb 56%  09 Feb 40%  02 Mar 100%  30 Mar 0%  06 Apr 45%  13 Apr 0%  20 Apr 50%  27 Apr 50%

2. by whether the map was played in 2025
   Breeze (not in 2025)          238 line-ups   top-1 67.4%   top-3 89.9%   from scratch, of 5 3.29
   maps played in 2025          1534 line-ups   top-1 65.4%   top-3 87.0%   from scratch, of 5 3.14


## What this means

*Written after the cells above ran, in a separate commit.*

**87.4% top-3 (range 86.1% to 88.6%) and 65.7% named first**, on 1,772 line-ups from a season it
was never tuned on. **By the rule set in advance, it beats the naive rule:** a lead of 15.3
points (14.0 to 16.6).

- **A little lower than late 2025, as expected:** 91.3% → 87.4%, and the naive rule 77.3% →
  72.1%.
- **The lead held** (14.0, then 15.3; the ranges overlap). The edge carries across a season
  boundary: the strongest evidence the project has.
- **The top three methods finish within about a point again** (87.4%, 88.2%, 86.8%).
- **From the map alone, the naive rule is still best** (3.30 of five against 3.16).
- **New agents are its blind spot:** Miks and Veto (77 of 8,860 questions) were in its top three
  42.9% of the time, and 0% in their first week.
- **A returning map was fine:** Breeze scored 89.9%.

It measures imitation, not winning. 2026's win results are still sealed.